In [ ]:
import os

# Set working directory to the root of your project
os.chdir("....")

# Confirm the working directory
print("✅ Current working directory:", os.getcwd())

# Ensure research folder exists
os.makedirs("research", exist_ok=True)


In [3]:
# 📦 Essential imports
import os
import re
import json
from tqdm import tqdm
from datetime import datetime
from dotenv import load_dotenv

from langchain.schema import Document
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Pinecone as LangchainPinecone
from langchain.chat_models import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import LLMChain


In [ ]:

# Load PDFs from a folder
def load_pdf_file(Data):
    loader = DirectoryLoader(Data, glob="*.pdf", loader_cls=PyPDFLoader)
    return loader.load()

# Clean noisy patterns from raw PDF text
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'(Page \d+|Figure \d+|Table \d+)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'(Copyright|Elsevier|Permissions|ISBN|Editor.*?Edition)', '', text, flags=re.IGNORECASE)
    return text.strip()

# Clean all loaded documents
def clean_documents(docs):
    cleaned = []
    for doc in docs:
        content = clean_text(doc.page_content)
        if len(content.strip()) > 50:
            cleaned.append(Document(page_content=content, metadata=doc.metadata))
    return cleaned

# Split documents into smaller chunks for embedding
def split_documents(docs, chunk_size=1000, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(docs)

# Load, clean, and split all at once
raw_docs = load_pdf_file("Data/")
cleaned_docs = clean_documents(raw_docs)
text_chunks = split_documents(cleaned_docs)

print(f"✅ Loaded {len(raw_docs)} raw docs")
print(f"✅ Cleaned {len(cleaned_docs)} docs")
print(f"✅ Split into {len(text_chunks)} chunks")


In [9]:
# ✅ Load HuggingFace BioBERT Embeddings
from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
)
print("✅ BioBERT embeddings model loaded")

# ✅ Setup Pinecone connection
import os
from dotenv import load_dotenv
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Connectting to Pinecone using existing index
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "cardicbot"  # Reusing already populated index

from langchain_pinecone import PineconeVectorStore

vectorstore = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

print(f"✅ Connected to existing Pinecone index: '{index_name}'")


✅ BioBERT embeddings model loaded
✅ Connected to existing Pinecone index: 'cardicbot'


In [16]:
import os
from langchain_openai import ChatOpenAI
from openai import OpenAIError
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# ✅ Step 1: RAG system prompt
system_prompt = (
    "You are a helpful and knowledgeable assistant specialized in cardiology and cardiovascular medicine. "
    "Use only the provided context from medical guidelines and textbooks to answer the question. "
    "If the answer is not contained in the context, reply with: 'I’m not sure based on the provided information.' "
    "Keep your answer medically accurate, concise (min 5 sentences, max 8-10 if needed), and avoid speculation.\n\nContext:\n{context}"
)


prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# ✅ Step 2: Check OpenAI API key and test LLM
openai_key = os.getenv("OPENAI_API_KEY")
assert openai_key is not None and len(openai_key) > 10, "❌ OPENAI_API_KEY is missing or too short!"

try:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    _ = llm.invoke("Say hello")  # Test call
    print("✅ OpenAI GPT-4o-mini model loaded and API key is valid")
except Exception as e:
    raise ValueError(f"❌ OpenAI API key might be invalid or expired:\n{str(e)}")

# ✅ Step 3: LangChain document combination (stuff strategy)
question_answer_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

# ✅ Step 4: Create retriever from Pinecone index
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})

# ✅ Step 5: Final RAG chain (retriever + QA chain)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("✅ RAG pipeline is ready")


✅ OpenAI GPT-4o-mini model loaded and API key is valid
✅ RAG pipeline is ready


In [17]:
from datetime import datetime

def ask_query_and_log(query, rag_chain, retriever, log_to_file=True, debug=False):
    """
    Run a query through the RAG pipeline and optionally log and debug it.

    Parameters:
    - query (str): The user question.
    - rag_chain (Runnable): The LangChain RAG chain object (retriever + QA).
    - retriever (BaseRetriever): The retriever to get relevant documents.
    - log_to_file (bool): If True, logs output and context to rag_audit_log.txt.
    - debug (bool): If True, prints retrieved chunks (source, page, preview) to console.
                    Useful for verifying which documents were used to generate the answer.

    Returns:
    - str: The final formatted answer + sources (also printed to console).
    """
    response = rag_chain.invoke({"input": query})
    answer = response["answer"]
    documents = response["context"]

    sources = []
    for doc in documents:
        source_file = doc.metadata.get("source", "Unknown Source").split("\\")[-1]
        page_number = int(doc.metadata.get("page", 0))
        sources.append(f"{source_file}, page {page_number}")

    formatted_sources = "\n".join([f"- {s}" for s in sources])
    final_output = f"Query: {query}\n\nAnswer:\n{answer}\n\nSources:\n{formatted_sources}"

    # Print to console
    print("\n" + final_output)

    # Build debug preview if requested
    preview_log = ""
    if debug:
        preview_log += f"\n======================\nRetrieved Chunks for: '{query}'\n======================\n"
        retrieved_docs = retriever.get_relevant_documents(query)
        for doc in retrieved_docs:
            source = doc.metadata.get("source", "Unknown Source").split("\\")[-1]
            page = doc.metadata.get("page", "Unknown Page")
            content = doc.page_content.strip()[:300].replace("\n", " ")
            preview_log += f"\n---\nSource: {source}, Page: {page}\nContent Preview:\n{content}\n"
        print(preview_log)

    # Optionally log to file
    if log_to_file:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        with open("research/rag_audit_log.txt", "a", encoding="utf-8") as f:
            f.write(f"\n\n{timestamp}\n{final_output}\n{preview_log}\n")

    return final_output


In [ ]:
ask_query_and_log("What is the role of beta blockers in managing heart failure?", rag_chain, retriever, debug=True)


In [19]:
from langchain.prompts import PromptTemplate
# ⛳️ Configuration
ENABLE_REFINEMENT = True                             # Optionally we can turn it off too
ALL_LOGS_FILE = "research/comparison_results.json"

# 🧠 LLM judge setup
EVAL_PROMPT_TEMPLATE = """
You are a medical cardio expert evaluating an assistant's answer to a clinical query. Use only the provided context.

Rate the answer from 1 to 5 (5 = excellent) on:

- Relevance
- Factual Accuracy
- Completeness
- Source Attribution
- Clarity

Then provide a one-line explanation and the average score.

---
Query: {query}

Context:
{context}

Answer:
{answer}

Evaluation Format:
Relevance: X  
Factual Accuracy: X  
Completeness: X  
Source Attribution: X  
Clarity: X  
Explanation: <your explanation>  
Average Score: X
"""

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def evaluate_answer(query, context_docs, answer):
    context_text = "\n\n".join([doc.page_content[:300] for doc in context_docs])
    eval_prompt = PromptTemplate.from_template(EVAL_PROMPT_TEMPLATE)
    prompt_input = eval_prompt.format(query=query, context=context_text, answer=answer)
    return judge_llm.invoke(prompt_input).content

def extract_score(text):
    scores = {}
    pattern = r"(Relevance|Factual Accuracy|Completeness|Source Attribution|Clarity):\s*([0-5](\.\d+)?)"
    for match in re.findall(pattern, text, re.IGNORECASE):
        scores[match[0].strip()] = float(match[1])

    avg_match = re.search(r"Average Score:\s*([0-5](\.\d+)?)", text)
    avg_score = float(avg_match.group(1)) if avg_match else None
    return scores, avg_score

# 🛠 Refinement setup
refinement_prompt = PromptTemplate.from_template("""
You are a senior medical editor. Improve the assistant's answer to the medical query below.

Focus on improving the following area(s) based on expert LLM feedback: {weak_dimensions}.

Ensure your answer is accurate, concise (≤8 sentences), context-based, and includes inline citations where appropriate. Do not add hallucinated content.

Query: {query}
Original Answer: {answer}

Sources:
{sources}

Refined Answer:
""")

refinement_chain = LLMChain(llm=llm, prompt=refinement_prompt)

def refine_answer_smart(query, answer, sources, weak_dimensions):
    weak_str = ", ".join(weak_dimensions)
    source_text = "\n".join([f"- {s['source']}, page {s['page']}" for s in sources])
    return refinement_chain.invoke({
        "query": query,
        "answer": answer,
        "sources": source_text,
        "weak_dimensions": weak_str
    })["text"]

# 🔁 Comparison loop
def compare_k_results_with_judge(query, k_values=[4, 6]):       # We can reduce it too
    local_log = []
    best_result = None
    highest_score = (-1, -1)  # avg, completeness

    for k in k_values:
        print(f"\n====================\nRunning with k = {k}\n====================")
        retrieved_docs = retriever.invoke(query, config={"k": k})

        # Deduplicate
        seen = set()
        unique_docs = []
        for doc in retrieved_docs:
            key = (doc.metadata.get("source"), doc.metadata.get("page"))
            if key not in seen:
                seen.add(key)
                unique_docs.append(doc)

        # Get answer
        input_payload = {"input": query, "context": unique_docs}
        result = question_answer_chain.invoke(input_payload)

        # Prepare source metadata
        sources = [{
            "source": doc.metadata.get("source", "Unknown").split("\\")[-1],
            "page": int(doc.metadata.get("page", 0))
        } for doc in unique_docs]

        # Judge
        evaluation = evaluate_answer(query, unique_docs, result)
        scores, avg = extract_score(evaluation)
        score_tuple = (avg, scores.get("Completeness", 0))

        print(f"\n📊 Evaluation for k={k}")
        print("Answer:\n", result)
        print("LLM Judge Feedback:\n", evaluation)
        print(f"📈 Scores: {scores}")
        print(f"🏁 Avg Score: {avg}")

        # Track best
        if score_tuple > highest_score:
            highest_score = score_tuple
            best_result = {
                "k": k,
                "answer": result,
                "score": avg,
                "dimension_scores": scores,
                "sources": sources
            }

        local_log.append({
            "query": query,
            "k": k,
            "answer": result,
            "llm_judge_feedback": evaluation,
            "llm_judge_score": {
                "dimension_scores": scores,
                "average": avg
            },
            "sources": sources,
            "refined_answer": None
        })

    # Refinement
    weak_areas = [k for k, v in best_result["dimension_scores"].items() if v < 5.0]
    refined_answer = None
    if ENABLE_REFINEMENT and weak_areas:
        print(f"\n🔧 Refining answer based on weaknesses: {weak_areas}...")
        refined_answer = refine_answer_smart(
            query, best_result["answer"], best_result["sources"], weak_areas
        )
        print("\n🧪 Refined Answer:\n", refined_answer)
    elif ENABLE_REFINEMENT:
        print("\n✅ No refinement needed. All scores are 5.0.")

    # Save refined result
    for entry in local_log:
        if entry["k"] == best_result["k"]:
            entry["refined_answer"] = refined_answer

    # Persist to file
    all_logs = []
    if os.path.exists(ALL_LOGS_FILE):
        with open(ALL_LOGS_FILE, "r", encoding="utf-8") as f:
            all_logs = json.load(f)

    all_logs.extend(local_log)
    with open(ALL_LOGS_FILE, "w", encoding="utf-8") as f:
        json.dump(all_logs, f, indent=2)

    print(f"\n🏆 Best k based on judge score: {best_result['k']} (Score: {best_result['score']})")
    print("\n📋 Final Best Answer:\n", best_result["answer"])
    if refined_answer:
        print("\n🪄 Refined Answer:\n", refined_answer)

    return best_result


In [ ]:
compare_k_results_with_judge("When is fibrinolysis contraindicated in acute myocardial infarction?")


In [ ]:
compare_k_results_with_judge("Compare ACE inhibitors and ARBs in the treatment of heart failure with reduced ejection fraction.")

In [ ]:
compare_k_results_with_judge("How is hypertrophic cardiomyopathy managed in symptomatic patients?")
